In [11]:
##利用SimpleDirectoryReader解析PDF文件
# from llama_index.core import SimpleDirectoryReader
# documents = SimpleDirectoryReader(input_files=["./datas/From Mind to Machine The Rise of Manus AI as a Fully.pdf"]).load_data()

# ##利用UnstructuredReader解析PDF文件
# from llama_index.readers.file.unstructured import UnstructuredReader
# from pathlib import Path

# reader=UnstructuredReader()
# documents=reader.load_data(file=Path("./datas/From Mind to Machine The Rise of Manus AI as a Fully.pdf"))

In [4]:
from llama_parse import LlamaParse
from llama_index.core import SimpleDirectoryReader, Settings,VectorStoreIndex,StorageContext
from llama_index.embeddings.dashscope import DashScopeEmbedding 
from llama_index.core.text_splitter import SentenceSplitter
import pickle  
from langchain.messages import HumanMessage
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.tools import tool
import dotenv
import os
dotenv.load_dotenv()

def init_llama_index_retriever():
    """初始化LlamaIndex的检索器（复用之前的逻辑）"""
    save_path = "./parsed_documents.pkl"
    pdf_path = "./datas/From Mind to Machine The Rise of Manus AI as a Fully.pdf"
    
    # 1. 加载/解析文档
    if os.path.exists(save_path):
        with open(save_path, 'rb') as f:
            documents = pickle.load(f)
    else:
        # 解析PDF
        parser = LlamaParse(
            api_key=os.getenv("LLAMA_PARSE_API_KEY"),
            result_type='markdown',
            verbose=True,
        )
        documents = SimpleDirectoryReader(
            input_files=[pdf_path],
            file_extractor={".pdf": parser},
        ).load_data()
        # 保存解析结果
        with open(save_path, 'wb') as f:
            pickle.dump(documents, f)
    
    Settings.embed_model = DashScopeEmbedding(
        model_name="text-embedding-v4",
        api_key=os.getenv("DASHSCOPE_API_KEY"),
        embed_batch_size=10,  
        timeout=60,
    )
    sentence_splitter = SentenceSplitter(chunk_size=512, chunk_overlap=20)
    
    index = VectorStoreIndex.from_documents(
        documents=documents,
        text_splitter=sentence_splitter,
        show_progress=True,
    )
    retriever = index.as_retriever(search_kwargs={"k":3})
    return retriever
llama_index_retriever = init_llama_index_retriever()

llm = init_chat_model(
    model='qwen3-max',
    model_provider='openai',
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv('DASHSCOPE_BASE_URL')
)

@tool
def document_retriever(query: str) -> str:
    """
    根据用户查询检索相关文档内容（核心工具函数）
    参数:
        query: 用户的查询问题（如"什么是Manus AI？"）
    返回:
        str: 检索到的相关文档内容，格式为字符串
    """
    # 使用初始化好的llama_index_retriever（避免同名冲突）
    try:
        retrieve_results = llama_index_retriever.retrieve(query)
        # 将检索结果转换为可读字符串
        result_text = ""
        for i, node in enumerate(retrieve_results):
            result_text += f"【相关内容{i+1}】\n{node.text}\n\n"
        return result_text if result_text else "未检索到相关内容"
    except Exception as e:
        return f"检索失败：{str(e)}"

agent = create_agent(
    model=llm,
    system_prompt="""你是一个专业的问答机器人，你需要先使用document_retriever工具检索相关文档，
    然后基于检索到的内容回答用户的问题。如果检索结果为空，直接告知用户未找到相关信息。""",
    tools=[document_retriever], 
)


Generating embeddings: 100%|██████████| 23/23 [00:06<00:00,  3.60it/s]


In [5]:
for step in agent.stream({'messages': HumanMessage('什么是Manus AI？')},stream_mode="values"):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

什么是Manus AI？
================================== Ai Message ==================================
Tool Calls:
  document_retriever (call_ad57b3a8ed8d4f1f867906b6)
 Call ID: call_ad57b3a8ed8d4f1f867906b6
  Args:
    query: 什么是Manus AI？
================================= Tool Message =================================
Name: document_retriever

【相关内容1】
# Unique Features and Capabilities

Through its architecture and training, Manus AI exhibits several unique features that distinguish it from conventional AI assistants:

- Autonomous Task Execution: Manus AI can carry out complex sequences of actions with minimal user intervention. Once given a high-level goal, it will plan, execute, and finalize the task largely on its own. This goes far beyond the typical AI, which would require the user to break down the problem or confirm each step. Manus “excels at various tasks in work and life, getting everything done while 